In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-pinecone \
    langchain-groq \
    pinecone \
    sentence-transformers \
    python-dotenv \
    pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_1775/3754768683.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Medical_book.pdf to Medical_book.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

def load_pdf_file(data):
  loader = PyPDFLoader(data)
  documents = loader.load()
  return documents

In [ ]:
import os

print(os.path.exists("Medical_book.pdf"))

True


In [ ]:
extracted_data = load_pdf_file("Medical_book.pdf")


In [ ]:
print(len(extracted_data))

637


In [ ]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [ ]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 5860


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings


In [ ]:
#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [ ]:
embedding = download_hugging_face_embeddings()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
query_result = embedding.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [ ]:
import os
os.environ["GROQ_API_KEY"] = "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"
os.environ["PINECONE_API_KEY"] ="pcsk_49dfar_SLb1jNR7JS1EgkCgBnLhCMsYByaBTTsNd8AvbMMimk1WZgEHjXo7cGy3aNXqM8M"

In [ ]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
GROQ_API_KEY=os.environ.get('GROQ_API_KEY')

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "medical-chatbot"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created successfully.")
else:
    print("Index already exists. Using existing index.")

Index created successfully.


In [ ]:
# Embed each chunk and upsert the embeddings into your Pinecone index.
# PineconeVectorStore: Connects LangChain with Pinecone to store and search document embeddings.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embedding,
)

In [ ]:
# Load Existing index

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [ ]:
docsearch

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("What is Acne?")

In [ ]:
retrieved_docs

[Document(id='fd255238-f6a7-4408-845b-ddc87b235553', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='f1a5eb5a-ee65-4a32-8485-eb626c214cd6', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 38.0, 'page_label': '39', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 04

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(
        model="openai/gpt-oss-20b",
        temperature=0
)

In [ ]:
# ChatPromptTemplate: Creates a structured prompt template for sending
# system instructions and user messages to the LLM.
from langchain_core.prompts import ChatPromptTemplate
# RunnablePassthrough: Passes the input data through a LangChain chain
# without changing it.
from langchain_core.runnables import RunnablePassthrough
# StrOutputParser: Converts the LLM's response into a simple string.
from langchain_core.output_parsers import StrOutputParser


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "Context:\n{context}"
)

# Creates a chat prompt containing a system message and a human/user message.
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
response = rag_chain.invoke("what is Acromegaly and gigantism?")
print(response)

Acromegaly and gigantism are disorders caused by excess growth hormone from the pituitary gland.  
Gigantism occurs when the excess hormone is present before the growth plates close, leading to excessive height.  
Acromegaly develops after the growth plates have fused, producing enlarged bones and soft tissues in adults.
